In [12]:
pip install faker

In [18]:
import sqlite3
import random
from faker import Faker
from datetime import datetime, timedelta
from tabulate import tabulate
from google.colab import files
from graphviz import Digraph

# SETUP dummy data generator
dummy = Faker('en_GB')
DB_NAME = "uk_solar_panel_monitoring_database_system.db"  # Shortened var name for casual use
NUM_CUSTOMERS = random.randint(1000, 1500)
NUM_ENGINEERS = 30

# Common values for duplicates
COMMON_PHONE = "07123456789"
COMMON_EMAIL = "info@solarsample.co.uk"

# CREATE DATABASE
def setup_db():
    # Open connection
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    cur.execute("PRAGMA foreign_keys = ON")  # Enable foreign key enforcement

    # Drop tables if they exist
    cur.executescript("""
        DROP TABLE IF EXISTS MaintenanceRecords;
        DROP TABLE IF EXISTS DailyReadings;
        DROP TABLE IF EXISTS WeatherData;
        DROP TABLE IF EXISTS SolarPanels;
        DROP TABLE IF EXISTS Installations;
        DROP TABLE IF EXISTS Customers;
        DROP TABLE IF EXISTS Engineers;
    """)

    # Customers table
    cur.execute("""
        CREATE TABLE Customers (
            customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            first_name TEXT NOT NULL,
            last_name TEXT NOT NULL,
            email TEXT,
            phone TEXT,
            address_line1 TEXT NOT NULL,
            city TEXT NOT NULL,
            postcode TEXT NOT NULL,
            country TEXT DEFAULT 'United Kingdom',
            customer_type TEXT CHECK(customer_type IN ('Residential','Commercial','Industrial')),
            registration_date TEXT,
            marketing_consent TEXT CHECK(marketing_consent IN ('Yes','No','Unknown')),
            notes TEXT
        )
    """)

    # Installations table
    cur.execute("""
        CREATE TABLE Installations (
            installation_id INTEGER PRIMARY KEY AUTOINCREMENT,
            customer_id INTEGER NOT NULL,
            installation_date TEXT,
            property_type TEXT,
            roof_type TEXT,
            installer_name TEXT,
            installation_cost REAL CHECK(installation_cost >= 0),
            FOREIGN KEY(customer_id) REFERENCES Customers(customer_id)
        )
    """)

    # Engineers table
    cur.execute("""
        CREATE TABLE Engineers (
            engineer_id INTEGER PRIMARY KEY AUTOINCREMENT,
            first_name TEXT,
            last_name TEXT,
            qualification_level TEXT CHECK(qualification_level IN ('NVQ','HNC','Degree')),
            years_experience INTEGER CHECK(years_experience >= 0),
            hourly_rate REAL CHECK(hourly_rate > 0),
            employment_status TEXT CHECK(employment_status IN ('Full-Time','Part-Time','Contract')),
            emergency_contact TEXT
        )
    """)

    # Solar Panels table
    cur.execute("""
        CREATE TABLE SolarPanels (
            panel_id INTEGER PRIMARY KEY AUTOINCREMENT,
            installation_id INTEGER,
            manufacturer TEXT,
            model_name TEXT,
            serial_number TEXT UNIQUE,
            capacity_kw REAL CHECK(capacity_kw > 0),
            orientation TEXT CHECK(orientation IN ('South','East','West')),
            inverter_brand TEXT,
            FOREIGN KEY(installation_id) REFERENCES Installations(installation_id)
        )
    """)

    # Weather Data table
    cur.execute("""
        CREATE TABLE WeatherData (
            panel_id INTEGER,
            reading_date TEXT,
            temperature_c REAL,
            sunshine_hours REAL,
            weather_condition TEXT,
            PRIMARY KEY(panel_id, reading_date),
            FOREIGN KEY(panel_id) REFERENCES SolarPanels(panel_id)
        )
    """)

    # Daily Readings table
    cur.execute("""
        CREATE TABLE DailyReadings (
            panel_id INTEGER,
            reading_date TEXT,
            energy_generated_kwh REAL,
            savings_pounds REAL,
            data_quality TEXT CHECK(data_quality IN ('Poor','Fair','Good','Excellent')),
            PRIMARY KEY(panel_id, reading_date),
            FOREIGN KEY(panel_id) REFERENCES SolarPanels(panel_id)
        )
    """)

    # Maintenance Records table
    cur.execute("""
        CREATE TABLE MaintenanceRecords (
            maintenance_id INTEGER PRIMARY KEY AUTOINCREMENT,
            panel_id INTEGER,
            engineer_id INTEGER,
            maintenance_date TEXT,
            maintenance_type TEXT CHECK(maintenance_type IN ('Routine','Emergency')),
            priority TEXT CHECK(priority IN ('Low','Medium','High','Critical')),
            cost_pounds REAL,
            FOREIGN KEY(panel_id) REFERENCES SolarPanels(panel_id),
            FOREIGN KEY(engineer_id) REFERENCES Engineers(engineer_id)
        )
    """)

    conn.commit()
    return conn

# Populate database with dummy data
def fill_data(conn):
    cur = conn.cursor()

    # Add customers (some missing emails)
    for _ in range(NUM_CUSTOMERS):
        title_choice = random.choice(["Mr", "Ms", "Mrs", None])
        # 10% missing email
        if random.random() > 0.9:
            email_val = None
        # 7% forced duplicate email
        elif random.random() < 0.07:
            email_val = COMMON_EMAIL
        else:
            email_val = dummy.email()

        # 8% forced duplicate phone number
        if random.random() < 0.08:
            phone_val = COMMON_PHONE
        else:
            phone_val = dummy.phone_number()

        notes_val = None if random.random() < 0.1 else dummy.sentence()

        cur.execute("""
            INSERT INTO Customers (
                title, first_name, last_name, email, phone, address_line1, city, postcode,
                customer_type, registration_date, marketing_consent, notes
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            title_choice,
            dummy.first_name(),
            dummy.last_name(),
            email_val,
            phone_val,
            dummy.street_address(),
            dummy.city(),
            dummy.postcode(),
            random.choice(["Residential", "Commercial", "Industrial"]),
            dummy.date_this_decade().isoformat(),
            random.choice(["Yes", "No", "Unknown"]),
            notes_val
        ))

    # Add engineers
    for _ in range(NUM_ENGINEERS):
        cur.execute("""
            INSERT INTO Engineers (
                first_name, last_name, qualification_level, years_experience, hourly_rate, employment_status
            ) VALUES (?, ?, ?, ?, ?, ?)
        """, (
            dummy.first_name(),
            dummy.last_name(),
            random.choice(["NVQ", "HNC", "Degree"]),
            random.randint(1, 30),
            round(random.uniform(20, 60), 2),
            random.choice(["Full-Time", "Part-Time", "Contract"])
        ))

    # Add installations and solar panels
    cur.execute("SELECT customer_id FROM Customers")
    all_customers = cur.fetchall()
    for cust_id_tuple in all_customers:
        cust_id = cust_id_tuple[0]
        inst_date = dummy.date_between(start_date='-5y', end_date='today').isoformat()
        cost_val = round(random.uniform(3000, 12000), 2)

        cur.execute("""
            INSERT INTO Installations (customer_id, installation_date, installation_cost)
            VALUES (?, ?, ?)
        """, (cust_id, inst_date, cost_val))
        inst_id = cur.lastrowid

        # Insert one solar panel per installation
        serial_num = dummy.unique.ean(length=13)
        cur.execute("""
            INSERT INTO SolarPanels (
                installation_id, manufacturer, model_name, serial_number, capacity_kw, orientation, inverter_brand
            ) VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            inst_id,
            random.choice(["LG", "Tesla", "SunPower"]),
            "SP-" + str(random.randint(1000, 9999)),
            serial_num,
            round(random.uniform(2.5, 6.0), 2),
            random.choice(["South", "East", "West"]),
            random.choice(["SMA", "Fronius", "SolarEdge"])
        ))

    # Generate weather data and daily readings for each panel
    cur.execute("SELECT panel_id FROM SolarPanels")
    panel_list = cur.fetchall()

    for panel_tuple in panel_list:
        panel_id = panel_tuple[0]
        # Generate roughly last 30 days of data
        for i in range(30):
            read_date = (datetime.now() - timedelta(days=i)).date().isoformat()
            temp = round(random.uniform(-2, 30), 1)
            sun_hours = round(random.uniform(0, 12), 1)
            weather_condition = random.choice(["Sunny", "Cloudy", "Rainy", "Partly Cloudy"])

            # Insert WeatherData
            cur.execute("""
                INSERT INTO WeatherData (
                    panel_id, reading_date, temperature_c, sunshine_hours, weather_condition
                ) VALUES (?, ?, ?, ?, ?)
            """, (panel_id, read_date, temp, sun_hours, weather_condition))

            # Energy calculation (rough estimate)
            base_capacity = 4  # average capacity
            energy = round(base_capacity * (sun_hours / 10) * random.uniform(0.8, 1.2), 2)
            savings = round(energy * 0.30, 2)

            # Insert DailyReadings
            cur.execute("""
                INSERT INTO DailyReadings (
                    panel_id, reading_date, energy_generated_kwh, savings_pounds, data_quality
                ) VALUES (?, ?, ?, ?, ?)
            """, (panel_id, read_date, energy, savings, random.choice(["Poor", "Fair", "Good", "Excellent"])))

    # Add maintenance records
    cur.execute("SELECT engineer_id FROM Engineers")
    eng_list = cur.fetchall()

    for panel_tuple in panel_list:
        panel_id = panel_tuple[0]
        # Randomly decide whether to add maintenance
        if random.random() > 0.6:
            engId = random.choice(eng_list)[0]
            maint_date = dummy.date_between(start_date='-2y', end_date='today').isoformat()
            maint_type = random.choice(["Routine", "Emergency"])
            base_cost = random.uniform(100, 300)
            if maint_type == "Emergency":
                base_cost *= 1.5
            # Insert maintenance record
            cur.execute("""
                INSERT INTO MaintenanceRecords (
                    panel_id, engineer_id, maintenance_date, maintenance_type, priority, cost_pounds
                ) VALUES (?, ?, ?, ?, ?, ?)
            """, (
                panel_id,
                engId,
                maint_date,
                maint_type,
                random.choice(["Low", "Medium", "High", "Critical"]),
                round(base_cost, 2)
            ))
    conn.commit()

# Function to display tables
def show_tables(conn, limit=20):
    cur = conn.cursor()
    tables = [
        "Customers",
        "Engineers",
        "Installations",
        "SolarPanels",
        "WeatherData",
        "DailyReadings",
        "MaintenanceRecords"
    ]
    for t in tables:
        print(f"\n--- {t.upper()} (First {limit} rows) ---")
        cur.execute(f"PRAGMA table_info({t})")
        headers = [c[1] for c in cur.fetchall()]
        cur.execute(f"SELECT * FROM {t} LIMIT {limit}")
        rows = cur.fetchall()
        print(tabulate(rows, headers=headers, tablefmt="grid"))

# Main execution
if __name__ == "__main__":
    conn = setup_db()
    fill_data(conn)
    show_tables(conn)
    print("\nDatabase file ready for download:")
    files.download(DB_NAME)


--- CUSTOMERS (First 20 rows) ---
+---------------+---------+--------------+-------------+---------------------------+--------------------+--------------------+---------------------+------------+----------------+-----------------+---------------------+---------------------+-----------------------------------------------------------------------+
|   customer_id | title   | first_name   | last_name   | email                     | phone              | address_line1      | city                | postcode   | country        | customer_type   | registration_date   | marketing_consent   | notes                                                                 |
+===============+=========+==============+=============+===========================+====================+====================+=====================+============+================+=================+=====================+=====================+=======================================================================+
|             1 | Ms      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>